In [ ]:
import torch
import torch.nn as nn

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

PyTorch version: 2.10.0+cu128
GPU available: True
Using device: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/beauty_rec/"
import os
os.makedirs(SAVE_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
!pip install -q "datasets==2.19.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 18.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


In [ ]:
import pandas as pd
from datasets import load_dataset
import os

CHUNK_SIZE = 50_000
OUT_PATH   = SAVE_DIR + "amazon_beauty_2023_full.csv"

print("Setting up streaming...")
reviews_stream = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_All_Beauty",
    split="full",
    streaming=True
)
print("Loading product metadata into memory (one-time, ~1.6M items)...")
meta_ds = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_meta_All_Beauty",
    split="full"
)
meta_df = meta_ds.to_pandas()
print(f"  Metadata loaded: {len(meta_df):,} products, "
      f"{meta_df.memory_usage(deep=True).sum() / 1e9:.2f} GB in RAM")

meta_df = meta_df.set_index("parent_asin")

print(f"\nStreaming reviews in chunks of {CHUNK_SIZE:,}...")
print(f"Writing to: {OUT_PATH}\n")

header_written = False
total_rows     = 0
chunk_buffer   = []
chunk_num      = 0

for record in reviews_stream:
    chunk_buffer.append(record)

    if len(chunk_buffer) >= CHUNK_SIZE:
        chunk_num += 1
        reviews_chunk = pd.DataFrame(chunk_buffer)
        chunk_buffer  = []
        merged = reviews_chunk.merge(
            meta_df.reset_index(),
            on="parent_asin",
            how="left",
            suffixes=("_review", "_meta")
        )

        merged.rename(columns={
            "title_x": "review_title",
            "title_y": "product_title",
        }, inplace=True)

        merged.to_csv(
            OUT_PATH,
            mode="a",
            index=False,
            header=not header_written
        )
        header_written = True
        total_rows    += len(merged)

        print(f"  Chunk {chunk_num:>4} written — "
              f"{len(merged):,} rows | "
              f"total so far: {total_rows:,} | "
              f"file size: {os.path.getsize(OUT_PATH)/1e9:.2f} GB")

if chunk_buffer:
    reviews_chunk = pd.DataFrame(chunk_buffer)
    merged = reviews_chunk.merge(
        meta_df.reset_index(),
        on="parent_asin",
        how="left",
        suffixes=("_review", "_meta")
    )
    merged.rename(columns={
        "title_x": "review_title",
        "title_y": "product_title",
    }, inplace=True)
    merged.to_csv(OUT_PATH, mode="a", index=False, header=not header_written)
    total_rows += len(merged)
    print(f"  Final flush — {len(merged):,} rows")

print(f"\n✅ Done! {total_rows:,} total rows → {OUT_PATH}")
print(f"   Final file size: {os.path.getsize(OUT_PATH)/1e9:.2f} GB")

Setting up streaming...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:1486: FutureWarning: The repository for McAuley-Lab/Amazon-Reviews-2023 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/McAuley-Lab/Amazon-Reviews-2023
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major re

Loading product metadata into memory (one-time, ~1.6M items)...


/usr/local/lib/python3.12/dist-packages/datasets/load.py:1486: FutureWarning: The repository for McAuley-Lab/Amazon-Reviews-2023 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/McAuley-Lab/Amazon-Reviews-2023
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating full split:   0%|          | 0/112590 [00:00<?, ? examples/s]

  Metadata loaded: 112,590 products, 0.16 GB in RAM

Streaming reviews in chunks of 50,000...
Writing to: /content/drive/MyDrive/beauty_rec/amazon_beauty_2023_full.csv

  Chunk    1 written — 50,000 rows | total so far: 50,000 | file size: 0.13 GB
  Chunk    2 written — 50,000 rows | total so far: 100,000 | file size: 0.26 GB
  Chunk    3 written — 50,000 rows | total so far: 150,000 | file size: 0.39 GB
  Chunk    4 written — 50,000 rows | total so far: 200,000 | file size: 0.52 GB
  Chunk    5 written — 50,000 rows | total so far: 250,000 | file size: 0.64 GB
  Chunk    6 written — 50,000 rows | total so far: 300,000 | file size: 0.77 GB
  Chunk    7 written — 50,000 rows | total so far: 350,000 | file size: 0.90 GB
  Chunk    8 written — 50,000 rows | total so far: 400,000 | file size: 1.03 GB
  Chunk    9 written — 50,000 rows | total so far: 450,000 | file size: 1.16 GB
  Chunk   10 written — 50,000 rows | total so far: 500,000 | file size: 1.29 GB
  Chunk   11 written — 50,000 ro

In [ ]:
import pandas as pd
import os

FILE = SAVE_DIR + "amazon_beauty_2023_full.csv"
sample = pd.read_csv(FILE, nrows=5)
print("\n===== FIRST 5 ROWS =====")
print(sample.head())
print("\n===== COLUMN NAMES =====")
print(sample.columns.tolist())
print("\n===== DATA TYPES =====")
print(sample.dtypes)


===== FIRST 5 ROWS =====
   rating                               title_review  \
0     5.0  Such a lovely scent but not overpowering.   
1     4.0     Works great but smells a little weird.   
2     5.0                                       Yes!   
3     1.0                          Synthetic feeling   
4     5.0                                         A+   

                                                text images_review  \
0  This spray is really nice. It smells really go...            []   
1  This product does what I need it to do, I just...            []   
2                          Smells good, feels great!            []   
3                                     Felt synthetic            []   
4                                            Love it            []   

         asin parent_asin                       user_id      timestamp  \
0  B00YQ6X8EO  B00YQ6X8EO  AGKHLEW2SOWHNMFQIJGBECAF7INQ  1588687728923   
1  B081TJ8YS3  B081TJ8YS3  AGKHLEW2SOWHNMFQIJGBECAF7INQ  15886158550

In [ ]:
import pandas as pd
FILE = SAVE_DIR + "amazon_beauty_2023_full.csv"
df_header = pd.read_csv(FILE, nrows=0)

row_count = 0
for chunk in pd.read_csv(FILE, chunksize=50_000, low_memory=False):
    row_count += len(chunk)
print("===== DATASET SHAPE =====")
print(f"Rows    : {row_count:,}")
print(f"Columns : {len(df_header.columns)}")

print("\n===== COLUMN NAMES =====")
for i, col in enumerate(df_header.columns, 1):
    print(f"{i}. {col}")

===== DATASET SHAPE =====
Rows    : 701,528
Columns : 25

===== COLUMN NAMES =====
1. rating
2. title_review
3. text
4. images_review
5. asin
6. parent_asin
7. user_id
8. timestamp
9. helpful_vote
10. verified_purchase
11. main_category
12. title_meta
13. average_rating
14. rating_number
15. features
16. description
17. price
18. images_meta
19. videos
20. store
21. categories
22. details
23. bought_together
24. subtitle
25. author


In [ ]:
import pandas as pd

FILE  = SAVE_DIR + "amazon_beauty_2023_full.csv"
OUT   = SAVE_DIR + "amazon_beauty_cleaned.csv"
CHUNK = 50_000

DROP_COLS    = ["bought_together", "author", "subtitle", "price", "videos"]
REQUIRED     = ["user_id", "parent_asin", "timestamp", "rating", "title_meta"]
EMPTY_VALS   = {"", "none", "null", "nan", "[]", "{}", "n/a", "na"}

header_written = False
total_in, total_out = 0, 0
for chunk in pd.read_csv(FILE, chunksize=CHUNK):
    total_in += len(chunk)
    chunk.drop(columns=[c for c in DROP_COLS if c in chunk.columns], inplace=True)

    for col in REQUIRED:
        if col not in chunk.columns:
            continue
        bad = chunk[col].isna() | chunk[col].astype(str).str.strip().str.lower().isin(EMPTY_VALS)
        chunk = chunk[~bad]

    chunk.to_csv(OUT, mode="a", header=not header_written, index=False)
    header_written = True
    total_out += len(chunk)

print(f"Input rows : {total_in:,}")
print(f"Output rows: {total_out:,}")
print(f"Dropped    : {total_in - total_out:,} ({100*(total_in - total_out)/max(total_in,1):.1f}%)")
print(f"Saved to   : {OUT}")

Input rows : 701,528
Output rows: 701,444
Dropped    : 84 (0.0%)
Saved to   : /content/drive/MyDrive/beauty_rec/amazon_beauty_cleaned.csv


In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict

FILE       = SAVE_DIR + "amazon_beauty_cleaned.csv"
OUT        = SAVE_DIR + "amazon_beauty_2M.csv"
CHUNK      = 50_000
TARGET     = 2_000_000
MIN_INTER  = 1
SEED       = 42

KEEP_COLS  = [
    "rating", "title_review", "text", "images_review",
    "asin", "parent_asin", "user_id", "timestamp", "helpful_vote",
    "main_category", "title_meta", "average_rating", "rating_number",
    "features", "description", "price", "images_meta", "categories", "details"
]
REQUIRED   = ["user_id", "parent_asin", "timestamp", "rating", "title_meta"]
EMPTY_VALS = {"", "none", "null", "nan", "[]", "{}", "n/a", "na"}

print("Pass 1: counting interactions...")
item_counts, user_counts = defaultdict(int), defaultdict(int)

for chunk in pd.read_csv(FILE, chunksize=CHUNK,
                         usecols=["user_id", "parent_asin"],
                         engine="python",
                         on_bad_lines="skip"):
    for k, v in chunk["parent_asin"].value_counts().items(): item_counts[k] += v
    for k, v in chunk["user_id"].value_counts().items():     user_counts[k] += v

valid_items = {k for k, v in item_counts.items() if v >= MIN_INTER}
valid_users = {k for k, v in user_counts.items() if v >= MIN_INTER}
print(f"  Valid items: {len(valid_items):,}  |  Valid users: {len(valid_users):,}")

print("Pass 2: cleaning + 1-core filter...")
chunks_out = []

for chunk in pd.read_csv(FILE, chunksize=CHUNK,
                         engine="python",
                         on_bad_lines="skip"):
    chunk = chunk[[c for c in KEEP_COLS if c in chunk.columns]]
    for col in REQUIRED:
        if col not in chunk.columns: continue
        bad = chunk[col].isna() | chunk[col].astype(str).str.strip().str.lower().isin(EMPTY_VALS)
        chunk = chunk[~bad]
    chunk = chunk[chunk["parent_asin"].isin(valid_items) &
                  chunk["user_id"].isin(valid_users)]
    chunk = chunk.sort_values("timestamp").drop_duplicates(
        subset=["user_id", "parent_asin"], keep="first"
    )

    if len(chunk): chunks_out.append(chunk)

df = pd.concat(chunks_out, ignore_index=True)
df = df.sort_values("timestamp") \
       .drop_duplicates(subset=["user_id", "parent_asin"], keep="first") \
       .reset_index(drop=True)

print(f"  After cleaning: {len(df):,} rows")
print("Step 3: stratified sampling to 2M...")

freq = df["parent_asin"].value_counts()
df["_label"] = df["parent_asin"].map(
    lambda a: "cold" if freq.get(a,0)==0 else ("few_shot" if freq.get(a,0)<=4 else "warm")
)

cold_df, few_df, warm_df = (df[df["_label"]==l] for l in ["cold","few_shot","warm"])
print(f"  Cold: {len(cold_df):,} | Few-shot: {len(few_df):,} | Warm: {len(warm_df):,}")

if len(df) <= TARGET:
    sampled = df
else:
    remaining = TARGET - len(cold_df)
    total_rest = len(few_df) + len(warm_df)
    few_n = min(int(remaining * len(few_df) / total_rest), len(few_df))
    warm_n = min(remaining - few_n, len(warm_df))
    sampled = pd.concat([
        cold_df,
        few_df.sample(n=few_n,   random_state=SEED),
        warm_df.sample(n=warm_n, random_state=SEED),
    ], ignore_index=True)

sampled = sampled.drop(columns=["_label"]) \
                 .sort_values("timestamp") \
                 .reset_index(drop=True)

print(f"  Final: {len(sampled):,} rows")
sampled.to_csv(OUT, index=False)
print(f"Saved → {OUT}")
print(sampled.dtypes)

Pass 1: counting interactions...
  Valid items: 112,553  |  Valid users: 631,915
Pass 2: cleaning + 1-core filter...
  After cleaning: 693,847 rows
Step 3: stratified sampling to 2M...
  Cold: 0 | Few-shot: 146,893 | Warm: 546,954
  Final: 693,847 rows
Saved → /content/drive/MyDrive/beauty_rec/amazon_beauty_2M.csv
rating            float64
title_review       object
text               object
images_review      object
asin               object
parent_asin        object
user_id            object
timestamp           int64
helpful_vote        int64
main_category      object
title_meta         object
average_rating    float64
rating_number       int64
features           object
description        object
images_meta        object
categories         object
details            object
dtype: object


In [ ]:
import pandas as pd
df = pd.read_csv(SAVE_DIR + "amazon_beauty_2M.csv")
df.head()

,rating,title_review,text,images_review,asin,parent_asin,user_id,timestamp,helpful_vote,main_category,title_meta,average_rating,rating_number,features,description,images_meta,categories,details
0,5.0,"The best electric toothbrush ever, REALLY!",We have used Oral-B products for 15 years; thi...,[],B000050FDE,B000050FDE,AED2GFGIAJ22PHMZGSKH2CPUF75Q,973052658000,10,All Beauty,Oral-B Professional Care 1000 Power Toothbrush,3.8,251,['Removes up to 97% of plaque from hard-to-rea...,['Product Description'\n 'The Oral-B Professio...,{'hi_res': array(['https://m.media-amazon.com/...,[],"{""Brand"": ""Oral-B"", ""Age Range (Description)"":..."
1,2.0,Fine while it's working,"I paid the full... for mine, but had to return...",[],B000050FDE,B000050FDE,AH54X3UMWTAMUJU2CVWYWNZVETLA,979657844000,20,All Beauty,Oral-B Professional Care 1000 Power Toothbrush,3.8,251,['Removes up to 97% of plaque from hard-to-rea...,['Product Description'\n 'The Oral-B Professio...,{'hi_res': array(['https://m.media-amazon.com/...,[],"{""Brand"": ""Oral-B"", ""Age Range (Description)"":..."
2,5.0,Over [price] for a toothbrush?? It's worth eve...,"First of all, I am not a dental professional.....",[],B000050AUD,B000050AUD,AHQGZITP7IEYTISSUELRAKFRH3GA,980874814000,24,All Beauty,Philips Sonicare PL-4 (4700) Sonic Toothbrush,3.4,50,[],"['Product Description'\n ""You could have a bet...","{'hi_res': array([None, None, None, None], dty...",[],"{""Brand"": ""PHILIPS"", ""Age Range (Description)""..."
3,5.0,Why did I wait so long?,"I admit it, I put off buying the Sonicare beca...",[],B000050AUD,B000050AUD,AEG7T4QNZ2EZEE4QVV6WZB2LXCOQ,983777277000,7,All Beauty,Philips Sonicare PL-4 (4700) Sonic Toothbrush,3.4,50,[],"['Product Description'\n ""You could have a bet...","{'hi_res': array([None, None, None, None], dty...",[],"{""Brand"": ""PHILIPS"", ""Age Range (Description)""..."
4,5.0,I wouldn't be without it.......,I purchased this tooth brush about five years ...,[],B000050AUD,B000050AUD,AHC4HE4WV6CUPI4K74KQ7YCPRPWA,985909127000,4,All Beauty,Philips Sonicare PL-4 (4700) Sonic Toothbrush,3.4,50,[],"['Product Description'\n ""You could have a bet...","{'hi_res': array([None, None, None, None], dty...",[],"{""Brand"": ""PHILIPS"", ""Age Range (Description)""..."


In [ ]:
df["timestamp"] = df["timestamp"].apply(
    lambda t: t // 1000 if t > 10_000_000_000 else t
)
print(pd.to_datetime(df["timestamp"], unit="s").describe())

count                           693847
mean     2019-04-08 19:21:35.274281472
min                2000-11-01 04:24:18
25%                2017-08-01 17:27:50
50%                2019-10-19 22:41:18
75%                2021-03-01 20:33:07
max                2023-09-09 00:39:36
Name: timestamp, dtype: object


In [ ]:
import pandas as pd

df = pd.read_csv(SAVE_DIR + "amazon_beauty_2M.csv")
print("=== Raw images_meta strings ===")
for i in range(5):
    print(f"\nRow {i}:")
    print(df["images_meta"].iloc[i])

print("\n=== Raw images_review strings ===")
for i in range(5):
    print(f"\nRow {i}:")
    print(df["images_review"].iloc[i])

=== Raw images_meta strings ===

Row 0:
{'hi_res': array(['https://m.media-amazon.com/images/I/61x2hmv6bVL._SL1500_.jpg',
       'https://m.media-amazon.com/images/I/81+8+WZ3WgL._SL1500_.jpg',
       'https://m.media-amazon.com/images/I/61V6q+-PuWL._SL1500_.jpg',
       'https://m.media-amazon.com/images/I/617P0k3I8jL._SL1500_.jpg',
       'https://m.media-amazon.com/images/I/61S30p1cseL._SL1500_.jpg'],
      dtype=object), 'large': array(['https://m.media-amazon.com/images/I/31AmKTPWMkL.jpg',
       'https://m.media-amazon.com/images/I/51sRPqnz3yL.jpg',
       'https://m.media-amazon.com/images/I/31rWh+qxmdL.jpg',
       'https://m.media-amazon.com/images/I/31s4pMkcvWL.jpg',
       'https://m.media-amazon.com/images/I/31eOvYwh+hL.jpg'],
      dtype=object), 'thumb': array(['https://m.media-amazon.com/images/I/31AmKTPWMkL._SS40_.jpg',
       'https://m.media-amazon.com/images/I/51sRPqnz3yL._SS40_.jpg',
       'https://m.media-amazon.com/images/I/31rWh+qxmdL._SS40_.jpg',
       'https:/

In [ ]:
import re

def extract_first_large_url(val):
    """Extract first 'large' URL using regex — bypasses the array() parsing issue"""
    urls = re.findall(r'https://m\.media-amazon\.com/images/[^\s\'"]+\.(?:jpg|gif|png)', str(val))
    return urls[0] if urls else None

# Test it
df["image_url"] = df["images_meta"].apply(extract_first_large_url)

print(f"Items WITH image URL: {df['image_url'].notna().sum():,} ({df['image_url'].notna().mean():.1%})")
print(f"Items WITHOUT image URL: {df['image_url'].isna().sum():,} ({df['image_url'].isna().mean():.1%})")

print("\nSample URLs:")
print(df["image_url"].dropna().head(5).tolist())

Items WITH image URL: 693,847 (100.0%)
Items WITHOUT image URL: 0 (0.0%)

Sample URLs:
['https://m.media-amazon.com/images/I/61x2hmv6bVL._SL1500_.jpg', 'https://m.media-amazon.com/images/I/61x2hmv6bVL._SL1500_.jpg', 'https://m.media-amazon.com/images/I/41RJNT098KL.jpg', 'https://m.media-amazon.com/images/I/41RJNT098KL.jpg', 'https://m.media-amazon.com/images/I/41RJNT098KL.jpg']


In [ ]:
unique_items = df[["parent_asin", "image_url", "description", "title_meta"]].drop_duplicates("parent_asin")
print(f"Unique items to embed: {len(unique_items):,}")

Unique items to embed: 112,553


In [ ]:
!pip install -q sentence_transformers

In [ ]:
import sys
print(sys.executable)

/usr/bin/python3


In [ ]:
import sys
!{sys.executable} -m pip install -q sentence_transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 116.7 MB/s eta 0:00:00


In [ ]:
import sys
!{sys.executable} -m pip install -q tf-keras

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import re
import requests
from io import BytesIO
from PIL import Image
from sentence_transformers import SentenceTransformer
from torchvision import models, transforms
import faiss
from tqdm import tqdm

df = pd.read_csv(SAVE_DIR + "amazon_beauty_2M.csv")

def extract_first_large_url(val):
    urls = re.findall(r'https://m\.media-amazon\.com/images/[^\s\'"]+\.(?:jpg|gif|png)', str(val))
    return urls[0] if urls else None

df["image_url"] = df["images_meta"].apply(extract_first_large_url)
items = (df[["parent_asin", "image_url", "description", "title_meta", "features", "categories"]]
           .drop_duplicates("parent_asin")
           .reset_index(drop=True))
items["item_idx"] = items.index
item_to_idx = dict(zip(items["parent_asin"], items["item_idx"]))
df["item_idx"] = df["parent_asin"].map(item_to_idx)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Unique items: {len(items):,}")
print(f"Device: {device}")

Unique items: 112,553
Device: cuda


In [ ]:
def build_text_input(row):
    parts = []
    for col in ["title_meta", "description", "features", "categories"]:
        val = str(row[col]) if pd.notna(row[col]) else ""
        val = re.sub(r"[\[\]']", "", val)
        parts.append(val.strip())
    return " ".join(p for p in parts if p)

items["text_input"] = items.apply(build_text_input, axis=1)
items["text_input"] = (
    items["text_input"]
    .str.lower()
    .str.replace(r"<.*?>", " ", regex=True)
    .str.replace(r"http\S+|www\S+", " ", regex=True)
    .str.replace(r"[^a-z0-9\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print(f"Empty text inputs: {(items['text_input'] == '').sum()}")
print(f"Sample: {items['text_input'].iloc[0][:200]}")

text_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
text_embeddings = text_model.encode(
    items["text_input"].tolist(),
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

print(f"Text embeddings shape: {text_embeddings.shape}")  # expect (224207, 384)
np.save(SAVE_DIR + "text_embeddings.npy", text_embeddings)
print("Saved: text_embeddings.npy")

Empty text inputs: 7
Sample: oral b professional care 1000 power toothbrush product description the oral b professional care 1000 offers a unique brush head that cups each tooth for a tooth by tooth clean to help prevent and even


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/440 [00:00<?, ?it/s]

Text embeddings shape: (112553, 384)
Saved: text_embeddings.npy


In [ ]:
from torchvision import models, transforms
import torch.nn as nn
import torch
import numpy as np
device = "cuda" if torch.cuda.is_available() else "cpu"
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import re
import requests
from io import BytesIO
from PIL import Image
from tqdm import tqdm

!pip install -q faiss-cpu
import faiss
df = pd.read_csv(SAVE_DIR + "amazon_beauty_2M.csv")

def extract_first_large_url(val):
    urls = re.findall(r'https://m\.media-amazon\.com/images/[^\s\'"]+\.(?:jpg|gif|png)', str(val))
    return urls[0] if urls else None

df["image_url"] = df["images_meta"].apply(extract_first_large_url)
items = (df[["parent_asin", "image_url", "description", "title_meta", "features", "categories"]]
           .drop_duplicates("parent_asin")
           .reset_index(drop=True))
items["item_idx"] = items.index
item_to_idx = dict(zip(items["parent_asin"], items["item_idx"]))
df["item_idx"] = df["parent_asin"].map(item_to_idx)

resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
resnet.fc = nn.Identity()
resnet = resnet.to(device)
resnet.eval()

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def download_and_transform(args):
    idx, path = args
    if not isinstance(path, str) or not path.startswith("http"):
        return idx, None
    try:
        response = requests.get(path, timeout=5)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content)).convert("RGB")
        return idx, transform(img)
    except Exception:
        return idx, None

image_embeddings = np.zeros((len(items), 2048), dtype="float32")
image_paths_enum = list(enumerate(items["image_url"].tolist()))

BATCH_SIZE = 64
batch_imgs, batch_idxs = [], []

with torch.no_grad():
    with ThreadPoolExecutor(max_workers=32) as executor:
        futures = {executor.submit(download_and_transform, arg): arg for arg in image_paths_enum}

        for future in tqdm(as_completed(futures), total=len(futures), desc="Image embeddings"):
            idx, img = future.result()
            if img is None:
                continue
            batch_imgs.append(img)
            batch_idxs.append(idx)

            if len(batch_imgs) == BATCH_SIZE:
                tensor = torch.stack(batch_imgs).to(device)
                embs = resnet(tensor).cpu().numpy()
                for k, i in enumerate(batch_idxs):
                    image_embeddings[i] = embs[k]
                batch_imgs, batch_idxs = [], []

        if batch_imgs:
            tensor = torch.stack(batch_imgs).to(device)
            embs = resnet(tensor).cpu().numpy()
            for k, i in enumerate(batch_idxs):
                image_embeddings[i] = embs[k]

faiss.normalize_L2(image_embeddings)
print(f"Image embeddings shape: {image_embeddings.shape}")
np.save(SAVE_DIR + "image_embeddings.npy", image_embeddings)
print("Saved: image_embeddings.npy")

Image embeddings: 100%|██████████| 112553/112553 [14:14<00:00, 131.70it/s] 


Image embeddings shape: (112553, 2048)
Saved: image_embeddings.npy
